In [53]:
%%time
%%capture log_imports
%reset -f
# !pip install --upgrade -q scikit-learn >> log    ;# upgrade SKL to v1.9 or later
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns # tensorflow_addons as tfa
from sklearn.preprocessing import PolynomialFeatures
import tensorflow as tf, tensorflow.keras as keras
from keras.layers import Flatten, Dense
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='ID') # rounds values to 2 decimals

CPU times: total: 453 ms
Wall time: 464 ms


In [54]:
df = pd.read_csv('XY_Shop.csv'); df

,Adm,AdmDur,Inf,InfDur,Prd,PrdDur,BncRt,ExtRt,PgVal,SpclDay,Mo,OS,Bsr,Rgn,TfcTp,VstTp,Wkd,Rev
0,0,0.00,0,0.00,18,132.99,0.038211,0.054523,0.000000,0.0,4,3,1,1,2,0,1,NaN
1,1,0.00,0,0.00,37,1150.20,0.001245,0.030321,0.000000,0.0,11,2,2,4,2,0,1,NaN
2,6,191.98,0,0.00,38,1266.78,0.004742,0.019551,17.816864,0.0,10,2,3,1,3,0,0,NaN
3,3,263.68,0,0.00,24,749.14,0.004474,0.024079,14.578547,0.0,7,2,2,2,1,0,1,NaN
4,0,0.00,0,0.00,3,136.41,0.000000,0.066300,0.000000,0.0,12,1,1,3,3,1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,14,262.46,0,0.00,170,3967.02,0.003314,0.015669,2.904034,0.0,12,2,2,4,2,0,0,0.0
499996,0,0.00,0,0.00,7,295.57,0.000000,0.000000,0.000000,0.0,9,1,5,4,3,0,0,0.0
499997,6,134.64,3,92.28,30,888.50,0.000000,0.003452,30.172020,0.0,11,2,2,1,4,0,0,0.0
499998,0,0.00,0,0.00,27,1185.14,0.000000,0.001593,0.000000,0.0,5,2,2,2,3,0,1,0.0


In [55]:
from sklearn.model_selection import train_test_split, StratifiedKFold,cross_validate,GroupKFold,cross_val_score,GridSearchCV,StratifiedGroupKFold,StratifiedShuffleSplit
from itertools import combinations, product
from sklearn.metrics import accuracy_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,PolynomialFeatures, OneHotEncoder,  FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline, clone
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [152]:
df_train = df[df.isnull()['Rev']==False]
df_train = df_train.drop_duplicates()
df_test = df.query('Rev!=Rev').drop('Rev', axis=1)

NumFeatures = list(df_train.select_dtypes(include='float').columns.drop(['Rev']))
OrdFeatures = ['Adm', 'Inf', 'Prd']
CatFeatures = list(df_train.columns.drop(NumFeatures+OrdFeatures).drop(['Rev']))


clean = True

if clean:
    #clean up outliers and unrealistic data
    lower = df_train[NumFeatures+OrdFeatures].quantile(0.00001)
    upper = df_train[NumFeatures+OrdFeatures].quantile(0.99999)

    mask = df_train[NumFeatures+OrdFeatures].ge(lower) & df_train[NumFeatures+OrdFeatures].le(upper)

    df_train_clean = df_train[mask.all(axis=1)]

X_clean = df_train_clean.drop(columns='Rev')
y_clean = df_train_clean['Rev']

In [57]:
display(NumFeatures,OrdFeatures,CatFeatures)

['AdmDur', 'InfDur', 'PrdDur', 'BncRt', 'ExtRt', 'PgVal', 'SpclDay']

['Adm', 'Inf', 'Prd']

['Mo', 'OS', 'Bsr', 'Rgn', 'TfcTp', 'VstTp', 'Wkd']

In [153]:
from sklearn.base import BaseEstimator, TransformerMixin

class ShopFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        eps = 1e-5

        other_added_col = []
        if 'SpclDay' in X.columns:
            X['SpclDay_B'] = (X['SpclDay'] > 0).astype(int)
            # other_added_col+=['SpclDay']
            
        # 1. Page Value flags & non-linear transform (strongest single signal)
        if 'PgVal' in X.columns:
            X['Val_is_zero'] = (X['PgVal'] == 0).astype(int)
            X['Val_log1p'] = np.log1p(np.maximum(0, X['PgVal']))
            # X['Val_log1p'] = np.log1p(X['PgVal'])
        # Skewed feature log-transforms
                
        skewed_cols = [
                        'Adm',
                        'Inf', 
                        'Prd',
                        'AdmDur', 
                        'InfDur', 
                        'PrdDur', 
                        # 'BncRt', 
                        # 'ExtRt', 
                        # 'PgVal',
                        ]
        
        for col in skewed_cols:
            # X[col + '_log'] = np.log1p(X[col])
            X[col + '_log'] = np.log1p(np.maximum(0,X[col]))
                        

        # 2. Bounce vs. Exit dynamics
        if 'BncRt' in X.columns and 'ExtRt' in X.columns:
            # X['Bnc_to_Ext_ratio'] = X['BncRt'] / (X['ExtRt'] + eps)
            X['Bnc_to_Ext_ratio_log'] = np.log1p(X['BncRt'] / (X['ExtRt'] + eps))
            X['Ext_minus_Bnc'] = X['ExtRt'] - X['BncRt']

        # 3. Time spent per page visited
        if 'PrdDur' in X.columns and 'Prd' in X.columns:
            X['Avg_Prd_Dur'] = X['PrdDur'] / (X['Prd'] + eps)
            X['Prd_Dur_log1p'] = np.log1p(np.maximum(0, X['PrdDur']))

        if 'AdmDur' in X.columns and 'Adm' in X.columns:
            X['Avg_Adm_Dur'] = X['AdmDur'] / (X['Adm'] + eps)

        if 'InfDur' in X.columns and 'Inf' in X.columns:
            X['Avg_Inf_Dur'] = X['InfDur'] / (X['Inf'] + eps)

        # 4. Total session engagement
        all_counts = [c for c in ['Adm', 'Inf', 'Prd'] if c in X.columns]
        all_durs = [c for c in ['AdmDur', 'InfDur', 'PrdDur'] if c in X.columns]

        if all_counts:
            X['Total_Pages'] = X[all_counts].sum(axis=1)
        if all_durs:
            X['Total_Duration'] = X[all_durs].sum(axis=1)
            X['Total_Duration_log1p'] = np.log1p(np.maximum(0, X['Total_Duration']))

        # 5. Intent progression proportions
        if 'Total_Pages' in X.columns:
            if 'Prd' in X.columns:
                X['Prd_Page_Share'] = X['Prd'] / (X['Total_Pages'] + eps)
            # if 'Adm' in X.columns:
            #     X['Adm_Page_Share'] = X['Adm'] / (X['Total_Pages'] + eps)
        X['PrdDur_Ratio'] = X['PrdDur'] / (X['Total_Duration'] + 1e-5)
        return X.drop(columns=skewed_cols+other_added_col)
        # return X
            

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", StandardScaler())
]).set_output(transform="pandas")

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output=False, categories='auto', handle_unknown="ignore"))
]).set_output(transform="pandas")

base_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NumFeatures + OrdFeatures),
        ("categorical", cat_pipeline, CatFeatures)
    ],
    verbose_feature_names_out=False
).set_output(transform="pandas")


In [ ]:
def build_wide_and_deep(num_dim, cat_cardinalities, num_crossed_buckets,seed=0):
    """
    num_dim: number of numerical dimensions
    cat_cardinalities: list of integer cardinalities (e.g., cat_dims = {'A':1,'B':2} for categorical features
    """
    # 1. Reset Keras/TF computational backend state
    keras.backend.clear_session()
    tf.random.set_seed(seed)

    # 2. Deterministic Initializers
    init_relu = keras.initializers.HeNormal(seed=seed)
    init_glorot = keras.initializers.GlorotNormal(seed=seed)
    init_embed = keras.initializers.RandomUniform(minval=-0.05, maxval=0.05, seed=seed)
    init_zeros = keras.initializers.Zeros()

    # --- INPUTS ---
    input_layer_num = layers.Input(shape=(num_dim,), name="num_input")
    input_layer_cats = [layers.Input(shape=(1,), dtype="int32", name=f"cat_{i}") for i in range(len(cat_cardinalities))]
    # input_layer_cats = [layers.Input(shape=(1,), dtype="int32", name=f"{k}") for k in cat_cardinalities.keys()]
        
    input_layer_crosses = layers.Input(shape=(num_crossed_buckets,), dtype="float32", name="crosses_input")

    # --- WIDE COMPONENT ---
    # 1. Linear pass over sparse categorical crosses and binary indicator features
    wide_linear = layers.Dense(1, use_bias=False, kernel_initializer=init_zeros, name="wide_crosses")(input_layer_crosses)
    
    # --- DEEP COMPONENT ---
    # 1. Normalize dense numerical features
    num_norm = layers.BatchNormalization()(input_layer_num)

    # 2. Embed categoricals
    cat_embeddings = []
    # for i, (input_cat, (cat,cardinaity)) in enumerate(zip(input_layer_cats, cat_cardinalities.items())):
    for i, (input_cat,cardinaity) in enumerate(zip(input_layer_cats, cat_cardinalities)):
        
        # Derive embedding dimension directly from each feature's cardinality
        emb_dim = max(2, min(50, int(cardinaity ** 0.5) + 2))
        emb = layers.Embedding(input_dim=cardinaity + 1, output_dim=emb_dim, 
                               embeddings_initializer=init_embed, name=f"emb_{i}")(input_cat)
        cat_embeddings.append(layers.Flatten()(emb))

    # 3. Concatenate and pass through MLP
    deep_in = layers.Concatenate()([num_norm] + cat_embeddings)

    # Block 1: 128 units
    h = layers.Dense(128, use_bias=False, name="dense_1")(deep_in)
    # h = layers.BatchNormalization(name="bn_1")(h)
    h = layers.Activation("relu", name="act_1")(h)
    # h = layers.Dropout(0.05)(h)

    # Block 2: 64 units
    h = layers.Dense(64, use_bias=False, name="dense_2")(h)
    # h = layers.BatchNormalization(name="bn_2")(h)
    h = layers.Activation("relu", name="act_2")(h)
    # h = layers.Dropout(0.1)(h)

    # Block 2: 32 units
    h = layers.Dense(32, use_bias=False, name="dense_3")(h)
    h = layers.BatchNormalization(name="bn_3")(h)
    h = layers.Activation("relu", name="act_3")(h)

    deep_linear = layers.Dense(1, kernel_initializer=init_glorot, name="deep_out")(h)

    # --- COMBINE LOGITS ---
    combined_logit = layers.Add()([wide_linear, deep_linear])
    out = layers.Activation("sigmoid", name="target")(combined_logit)

    model = keras.Model(inputs=[input_layer_num, input_layer_crosses] + input_layer_cats, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [220]:
class CardinalShift1(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        X = X.copy()
        X[['VstTp', 'Wkd']]=X[['VstTp', 'Wkd']]+1
        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            if not hasattr(self, "feature_names_in_"):
                raise ValueError(
                    "input_features must be provided when fitting on "
                    "data without column names."
                )
            input_features = self.feature_names_in_

        input_features = np.asarray(input_features, dtype=object)

        return input_features

class inter_feature_crosses(BaseEstimator, TransformerMixin):
    def __init__(self, cross_pairs):
        super().__init__()
        self.cross_pairs = cross_pairs
        self.GENERATED_FEATURES = [f"{col1}_x_{col2}" for col1, col2 in cross_pairs]

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for col1, col2 in self.cross_pairs:
            X[f"{col1}_x_{col2}"] = X[col1].astype(str) + "_" + X[col2].astype(str)
        return X
        
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            if not hasattr(self, "feature_names_in_"):
                raise ValueError(
                    "input_features must be provided when fitting on "
                    "data without column names."
                )
            input_features = self.feature_names_in_

        input_features = np.asarray(input_features, dtype=object)

        return np.concatenate([
                    input_features,
                    np.asarray(self.GENERATED_FEATURES, dtype=object),
                ])

def prepare_model_inputs(df, num_cols, wide_cols, cat_cols):
    """
    Prepares a clean list of NumPy arrays matching the exact Keras inputs:
    [num_tensor, wide_tensor, cat_tensor_1, cat_tensor_2, ...]
    """
    x_num = df[num_cols].to_numpy(dtype='float32')
    x_wide = df[wide_cols].to_numpy(dtype='float32')
    
    # Extract each categorical column in the exact order of cat_dict keys
    x_cats = [df[col].to_numpy(dtype='int32').reshape(-1, 1) for col in cat_cols]
    
    return [x_num, x_wide] + x_cats

seeds = [1,10,20,30,42,]
accu = []
f1 = []

for s in seeds:
    X_clean_train, X_clean_val, y_clean_train, y_clean_val = train_test_split(
        X_clean,
        y_clean,
        train_size=0.8,
        stratify=y_clean,
        shuffle=True,
        random_state=s
    )

    cross_cat_cols_1 = ['Mo', 'VstTp', 'Wkd']
    cross_pairs = [('Mo', 'VstTp'), ('Wkd', 'VstTp'), ('Mo', 'Wkd')]
    cross_cat_cols_2 = ['Val_is_zero','SpclDay_B']


    fe = ShopFeatureEngineer()

    # whole set
    # X_fe = fe.fit_transform(X_clean)

    # whole train set
    X_fe = fe.fit_transform(X_clean_train)

    NumFeaturesNew = list(X_fe.columns.drop(CatFeatures+cross_cat_cols_2))
    NewCatFeatures = CatFeatures +cross_cat_cols_2
    X_fe = X_fe.reindex(columns=NumFeaturesNew+NewCatFeatures)

    cross_pipeline = Pipeline([
        ('cross', inter_feature_crosses(cross_pairs=cross_pairs)),
        ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    ]).set_output(transform="pandas")

    final_process = ColumnTransformer(
        transformers=[
            ("shift1", CardinalShift1(), CatFeatures),
            ("cross_ohe", cross_pipeline, cross_cat_cols_1),
            ("scaler",StandardScaler(), NumFeaturesNew)
        ],
        remainder="passthrough",
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    X_final_transformed = final_process.fit_transform(X_fe)
    WideFeatures = X_final_transformed.columns.drop(NewCatFeatures+NumFeaturesNew).tolist()+cross_cat_cols_2

    num_crossed_buckets = len(WideFeatures)

    X_final_transformed = X_final_transformed.reindex(columns=NumFeaturesNew+WideFeatures+CatFeatures)

    cat_cardinalities_raw = {
        col: int(X_final_transformed[col].max()) for col in CatFeatures
    }
    cat_cardinalities = [cat_cardinalities_raw[k] for k in CatFeatures]

    model = build_wide_and_deep(
        num_dim=len(NumFeaturesNew),
        cat_cardinalities=cat_cardinalities,
        num_crossed_buckets=num_crossed_buckets
    )

    #validation set
    Xv_fe = fe.transform(X_clean_val)
    Xv_fe = Xv_fe.reindex(columns=NumFeaturesNew+NewCatFeatures)
    X_clean_val_trans = final_process.transform(Xv_fe)
    X_clean_val_trans = X_clean_val_trans.reindex(columns=NumFeaturesNew+WideFeatures+CatFeatures)

    # tf.keras.utils.set_random_seed(0)

    # Init = keras.initializers.RandomNormal(seed=0)

    # Format Train & Validation Inputs
    X_train_inputs = prepare_model_inputs(
        X_final_transformed, 
        NumFeaturesNew, 
        WideFeatures, 
        CatFeatures
    )

    X_val_inputs = prepare_model_inputs(
        X_clean_val_trans, 
        NumFeaturesNew, 
        WideFeatures, 
        CatFeatures
    )

    model.fit(
        X_train_inputs,
        y_clean_train.values,
        # y_clean.values,

        epochs=30,
        batch_size=2048,
        verbose=0
    )

    y_pred = model.predict(X_val_inputs, verbose=0)

    accu += [accuracy_score(y_clean_val, (y_pred >= 0.5).astype(int))]
    f1 += [f1_score(y_clean_val, (y_pred >= 0.5).astype(int))]

print(f'{np.mean(accu):.3f},', end="")
print(f'{np.mean(f1):.3f}')


0.950,0.881
